# LoL Scrape

Pulls gol.gg team, player, champion, and game data for LCK/LPL/LEC/LCS
and caches it into dated data_cache snapshots. Run this first; every
other notebook reads its cached output tables. See
docs/superpowers/specs/2026-09-06-lol-scrape-design.md for the design
this notebook implements.

Set TEST_MODE = True for a first run: caps to one region and its first
2 teams so the whole path (list tables, team matchlist, game manifest,
game parse, hash guard, snapshot write) can be checked cheaply before a
full run.

In [1]:
from __future__ import annotations

import csv
from pathlib import Path

import pandas as pd
import requests

import lol_lib
import lol_scrape_lib

## 2. Parameters

In [2]:
SEASON = "S16"
SPLIT = "Summer"
TEST_MODE = True
TEST_MODE_TEAM_LIMIT = 2

REGIONS = lol_lib.REGIONS if not TEST_MODE else lol_lib.REGIONS[:1]

## 3. Session setup

In [3]:
lol_lib.print_versions()
lol_lib.setup_output_dirs()
session = requests.Session()

Package versions:
python: 3.14.5
pandas: 2.3.3
numpy: 2.4.6
matplotlib: 3.11.0
requests: 2.34.2


## 4. Global lists (players, champions)

In [4]:
players_df = lol_scrape_lib.scrape_global_list("players", SEASON, SPLIT, session)
champions_df = lol_scrape_lib.scrape_global_list("champion", SEASON, SPLIT, session)

global_tables = {"players": players_df, "champions": champions_df}
latest_global = lol_lib.latest_global_scrape_snapshot()

if latest_global is not None and all(
    lol_scrape_lib.hash_table(df) == lol_scrape_lib.hash_table(
        pd.read_csv(latest_global / f"{name}.csv", keep_default_na=False)
    )
    for name, df in global_tables.items()
    if (latest_global / f"{name}.csv").exists()
):
    print(f"no changes to global lists since {latest_global.name}, skipping snapshot")
else:
    snapshot_dir = lol_lib.global_scrape_snapshot_dir()
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    for name, df in global_tables.items():
        df.to_csv(snapshot_dir / f"{name}.csv", index=False)
    print(f"wrote global snapshot to {snapshot_dir}")

no changes to global lists since 2026-09-06, skipping snapshot


## 5. Per-region teams and games

## 6. Resumable game fetch

In [5]:
for region in REGIONS:
    print(f"--- {region} ---")

    teams_df = lol_scrape_lib.scrape_region_teams(region, SEASON, SPLIT, session)
    if TEST_MODE:
        teams_df = teams_df.head(TEST_MODE_TEAM_LIMIT)

    latest_region = lol_lib.latest_scrape_snapshot(region)
    teams_unchanged = (
        latest_region is not None
        and (latest_region / "teams.csv").exists()
        and lol_scrape_lib.hash_table(teams_df) == lol_scrape_lib.hash_table(
            pd.read_csv(latest_region / "teams.csv", keep_default_na=False)
        )
    )

    snapshot_dir = lol_lib.scrape_snapshot_dir(region)
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    games_path = snapshot_dir / "games.csv"
    failures_path = snapshot_dir / "failures.csv"

    already_fetched_ids: set[str] = set()
    if games_path.exists():
        already_fetched_ids = set(pd.read_csv(games_path)["game_id"].astype(str))

    def append_row(row: dict, path: Path = games_path) -> None:
        is_new_file = not path.exists()
        with open(path, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(row.keys()))
            if is_new_file:
                writer.writeheader()
            writer.writerow(row)

    def append_failure(failure: dict, path: Path = failures_path) -> None:
        is_new_file = not path.exists()
        with open(path, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["game_id", "url", "error"])
            if is_new_file:
                writer.writeheader()
            writer.writerow(failure)

    lol_scrape_lib.scrape_region_games(
        region, SEASON, SPLIT, teams_df, session,
        already_fetched_ids, on_row=append_row, on_failure=append_failure,
    )

    if not teams_unchanged or games_path.exists():
        teams_df.to_csv(snapshot_dir / "teams.csv", index=False)
    print(f"finished {region}: snapshot at {snapshot_dir}")

--- LCK ---


finished LCK: snapshot at C:\Users\adamh\Documents\LoL_Esports\.claude\worktrees\lol-scrape-notebook\data_cache\LCK\2026-09-06
